In [22]:
# Dependencies
import json
import os

from sklearn.model_selection import train_test_split

In [23]:
# Train-Test split
data = "data.json"
with open(data,"r") as file:
    data = json.load(file)

train_data,val_data = train_test_split(data, test_size=0.1, random_state=42)

# Dump to train.json
train = "train.json"
with open(train,"w") as f:
    json.dump(train_data, f, indent=2)
# Dump to val.json
val = "val.json"
with open(val,"w") as f:
    json.dump(val_data, f, indent=2)

In [21]:
# Stage 3: Training
from transformers import TrainingArguments, Trainer
from torch.utils.data import DataLoader

# Training arguments
training_args = TrainingArguments(
    output_dir="./llava-med-finetuned",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,    # effective batch = 4 * 4 = 16
    warmup_ratio=0.03,
    learning_rate=2e-4,               # higher LR is OK for LoRA
    lr_scheduler_type="cosine",
    bf16=True,                        # use bfloat16
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=200,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="tensorboard",          # or "wandb"
    dataloader_num_workers=4,
    remove_unused_columns=False       # critical — keep pixel_values
)
    
# Processor
processor = AutoProcessor.from_pretrained(model_path)
# Dataset
train_dataset = LLaVaMedDataset("train.json", "", processor, tokenizer)
val_dataset = LLaVaMedDataset("val.json", "", processor, tokenizer)


# Define compute_metrics
import numpy as np
import evaluate
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

def compute_metrics(eval_preds):
    predictions,label = eval_preds
    
    pred_ids  = np.argmax(predictions[0], axis=-1) if isinstance(predictions, tuple) else predictions
    label_ids = np.where(labels != -100, labels, tokenizer.pad_token_id)
    # Decode
    decoded_pred = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    decoded_label = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]
    
    # Calculate BLEU and ROUGE
    bleu_score  = bleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    rouge_score = rouge.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "bleu":   bleu_score["bleu"],
        "rouge1": rouge_score["rouge1"],
        "rougeL": rouge_score["rougeL"],
    }

train = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    compute_metrics = compute_metrics
)

# train.train()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [17]:
import inspect

processorr = AutoProcessor.from_pretrained(model_path)
print(inspect.signature(processorr))

(images: Union[ForwardRef('PIL.Image.Image'), numpy.ndarray, ForwardRef('torch.Tensor'), list['PIL.Image.Image'], list[numpy.ndarray], list['torch.Tensor'], NoneType] = None, text: str | list[str] | list[list[str]] = None, **kwargs: typing_extensions.Unpack[transformers.models.llava.processing_llava.LlavaProcessorKwargs]) -> transformers.feature_extraction_utils.BatchFeature
